### DPA-DP phase switch validation on CIFAR-10

To validate the DPA-DP phase-switch idea independently of the AIDER dataset's size constraints, 
the same experimental design was repeated on CIFAR-10, which is large enough to avoid the duplication 
and epsilon overflow issues encountered with AIDER. Since CIFAR-10 has no disaster classes,
the disaster phase was simulated by introducing two new classes during training.

**Class selection.** The two classes absent during the normal phase were "bird" (class 2) and "deer" (class 4),
chosen as the third and fourth most difficult classes in CIFAR-10 after the notoriously hard "cat" and "dog" classes [1, 2, 3].
The intent is to approximate the AIDER setup, where disaster imagery is rare and visually harder for the model than everyday imagery, by using classes that are already known to be difficult for CIFAR-10 classifiers, rather than selecting classes at random.

**Phase construction.** Each round uses a fixed total of 40,000 training images, redistributed across phases so that the two withheld classes are entirely absent during the normal phase and re-introduced at disaster onset:

- **Normal phase:** 5,000 images each from the 8 non-withheld classes, 0 images from "bird" and "deer" (5000, 5000, 0, 5000, 0, 5000, 5000, 5000, 5000, 5000 = 40,000).
- **Disaster phase:** 3,750 images each from the 8 non-withheld classes plus 5,000 each from "bird" and "deer" (3750, 3750, 5000, 3750, 5000, 3750, 3750, 3750, 3750, 3750 = 40,000).

This mirrors the AIDER setup's structure - an extended period in which the model never sees the target classes, followed by their sudden appearance at a fixed phase-transition round - while keeping the per-round image count constant across both phases.

This design tests the DPA-DP idea as a general claim: that a phase-aware privacy budget helps a federated model 
performance on previously-unseen classes, independent of whether they represent a real-world disaster.

1. Y. Abouelnaga, O. S. Ali, H. Rady, and M. Moustafa, "CIFAR-10: KNN-Based Ensemble of Classifiers," in 2016 International Conference on Computational Science and Computational Intelligence (CSCI), pp. 1192-1195, IEEE.
2. N. K. Erkek, E. Balci, and B. Halay, "Experimental Analysis of Neural Network-Based Image Classification on the CIFAR-10 Dataset."
3. P. K. Mannepalli and S. S. Kang, "A CNN-Based Image Classification Framework with Explainable AI Analysis on the CIFAR-10 Datase," in 2026 World Conference on Computational Science and Technology (WcCST), pp. 815-820, IEEE

##### page break

In [ ]:
import pandas
from matplotlib import pyplot
import numpy
def confusion_matrix(profile, ax0=None):
    data = pandas.read_csv("confusion.csv")
    data["confusion"] = data["confusion"].apply(eval)
    cm0 = data[(data["id"] == profile) & (data["seed"] == 0)].iloc[0]["confusion"]
    cm1 = data[(data["id"] == profile) & (data["seed"] == 1)].iloc[0]["confusion"]
    cm2 = data[(data["id"] == profile) & (data["seed"] == 2)].iloc[0]["confusion"]
    cm = numpy.rint(numpy.mean([cm0, cm1, cm2], axis=0)).astype(int)
    ax = pyplot.subplots(figsize=(5, 4))[1] if ax0 is None else ax0
    image = ax.imshow(cm, cmap="Blues")
    for row in range(len(cm)):
        for col in range(len(cm[row])):
            ax.text(col, row, cm[row][col], ha="center", va="center",
                color="white" if cm[row][col] > max(max(row) for row in cm) / 2 else "black")
    ax.set_title("confusion matrix for " + profile + " profile")
    ax.set_xticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
    ax.set_yticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
    ax.set_xticklabels(["a", "a", "b", "c", "d", "d", "f", "h", "s", "t"])
    ax.set_yticklabels(["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"])
    if ax0 is None:
        pyplot.tight_layout()
        pyplot.show()

In [ ]:
def fixed_diff(data):
    fixed = (data[data["id"] == "fixed"][["seed", "round", "accuracy", "loss"]]
        .rename(columns={"accuracy": "fixed_accuracy", "loss": "fixed_loss",}))
    data_diff = data.merge(fixed, on=["seed", "round"])
    data_diff["accuracy"] = data_diff["accuracy"] - data_diff["fixed_accuracy"]
    data_diff["loss"] = data_diff["loss"] - data_diff["fixed_loss"]
    data_diff = data_diff.drop(columns=["fixed_accuracy", "fixed_loss"])
    return data_diff

In [ ]:
import pandas
from matplotlib import pyplot
def flwr_accuracy_error(file, start=0, key="accuracy"):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] >= start]
    data = fixed_diff(data)
    summary = (data.groupby(["id", "round"], sort=False)[key].agg(mean="mean", std="std").reset_index())
    pyplot.figure(figsize=(11, 6))
    for id_val, group in summary.groupby("id", sort=False):
        group = group.sort_values("round")
        if id_val == "fixed":
            pyplot.plot(group["round"], group["mean"], label=str(id_val))
        else:
            pyplot.errorbar(group["round"], group["mean"], yerr=group["std"], linewidth=2, capsize=4, alpha=0.7, label=str(id_val))
    pyplot.xlabel("round")
    pyplot.ylabel(key)
    pyplot.grid(True, alpha=0.3)
    pyplot.legend()
    pyplot.tight_layout()
    pyplot.show()

In [ ]:
import pandas
from matplotlib import pyplot
def flwr_accuracy(file, start=0, key="accuracy"):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] >= start]
    summary = (data.groupby(["id", "round"], sort=False)[key].agg(mean="mean", std="std").reset_index())
    figure = pyplot.figure(figsize=(11, 6))
    for id_val, group in summary.groupby("id", sort=False):
        group = group.sort_values("round")
        line, = pyplot.plot(group["round"], group["mean"], label=str(id_val))
        pyplot.fill_between(
        group["round"],
        group["mean"] - group["std"],
        group["mean"] + group["std"],
        color=line.get_color(),
        alpha=0.2
    )

    pyplot.xlabel("round")
    pyplot.ylabel(key)
    pyplot.grid(True, alpha=0.3)
    pyplot.legend()
    pyplot.tight_layout()
    pyplot.show()
    # figure.savefig(file + "_" + key + "_diagram.pdf", bbox_inches="tight")
flwr_accuracy("no_phase/no_phase", 160, "loss")

In [ ]:
def accuracy_summary_1(file):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] > 190]
    summary = (data.groupby("id", sort=False)["accuracy"].agg(mean="mean", deviation="std").reset_index())
    fixed_mean = summary.loc[summary["id"] == "fixed", "mean"].iloc[0]
    summary["diff"] = summary["mean"] - fixed_mean
    # return summary
    data_sub = pandas.read_csv(file + "_sub.csv")
    data_sub = data_sub[data_sub["round"] > 190]
    summary_sub = (data_sub.groupby("id", sort=False)["accuracy"].agg(mean_g="mean", dev_g="std").reset_index())
    fixed_mean_sub = summary_sub.loc[summary["id"] == "fixed", "mean_g"].iloc[0]
    summary_sub["diff_g"] = summary_sub["mean_g"] - fixed_mean_sub
    summary = summary.merge(summary_sub, on="id")
    summary = summary.set_index("id")
    summary.index.name = None
    summary = summary.style.format("{:.4f}")
    return summary
accuracy_summary_1("test3/flwr")
# accuracy_summary_1("no_phase/no_phase")

In [ ]:
def accuracy_summary(file):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] > 190]
    fixed = (
        data[data["id"] == "fixed"]
        [["seed", "round", "accuracy"]]
        .rename(columns={"accuracy": "fixed"})
    )
    data = data.merge(fixed, on=["seed", "round"], how="left")
    data["fixed"] = data["accuracy"] - data["fixed"]
    summary = (data.groupby("id", sort=False)["accuracy"].agg(mean="mean", deviation="std").reset_index())
    summary_diff = (data.groupby("id", sort=False)["fixed"].agg(diff="mean", dev_diff="std").reset_index())
    summary = summary.merge(summary_diff, on=["id"], how="left")
    summary = summary.set_index("id")
    summary.index.name = None
    summary = summary.style.format("{:.4f}")
    return summary
accuracy_summary("test3/flwr")
# accuracy_summary2("no_phase/no_phase")

In [ ]:
from matplotlib import pyplot
import opacus.accountants

delta = 1e-5

def epsilon_curve(noise_multiplier_1, noise_multiplier_2, accountant):
    filename = f"epsilon_{noise_multiplier_1:.2f}_{noise_multiplier_2:.2f}_{noise_multiplier_switch}_{rounds}_{batch_size}_{n_samples}.txt"
    try:
        with open(filename) as file:
            return [float(line) for line in file]
    except FileNotFoundError:
        pass
    n_batches = (n_samples + batch_size - 1) // batch_size
    sample_rate = 1 / n_batches
    epsilons = [0]
    for round_idx in range(rounds):
        noise_multiplier = noise_multiplier_1 if round_idx < noise_multiplier_switch else noise_multiplier_2
        for _ in range(n_batches):
            accountant.step(noise_multiplier=noise_multiplier, sample_rate=sample_rate)
        epsilons.append(float(accountant.get_epsilon(delta=delta)))
    with open(filename, "w") as file:
        for value in epsilons:
            file.write(f"{value}\n")
    return epsilons

def plot_epsilon(data):
    figure = pyplot.figure(figsize=(13,4))
    for name, arr in data.items():
        pyplot.plot(arr, label=name)
    pyplot.xlabel("round")
    pyplot.ylabel("epsilon")
    pyplot.grid(True, alpha=0.3)
    pyplot.legend()
    pyplot.show()
    # figure.savefig("epsilon.pdf", bbox_inches="tight")

##### page break

### DPA-DP privacy budget allocation
Epsilon budget consumption over 200 rounds (180 normal + 20 simulated disaster phase), for four noise-multiplier strategies tuned to reach a matched final budget of epsilon around 8.1.
- fixed - uniform noise multiplier 1.0 across both phases
- dpa-dp-0.85 - low profile: 1.03 normal, 0.85 disaster
- dpa-dp-0.75 - medium profile: 1.09 normal, 0.75 disaster
- dpa-dp-0.65 - high profile: 1.53 normal, 0.65 disaster, the largest phase difference


In [ ]:
rounds=200
noise_multiplier_switch = 180
n_samples = 4000
batch_size = 32

epsilon_data = {}
epsilon_data["fixed"] = epsilon_curve(1.00, 1.00, opacus.accountants.PRVAccountant())
epsilon_data["dpa-dp-0.85"] = epsilon_curve(1.03, 0.85, opacus.accountants.PRVAccountant())
epsilon_data["dpa-dp-0.75"] = epsilon_curve(1.09, 0.75, opacus.accountants.PRVAccountant())
epsilon_data["dpa-dp-0.65"] = epsilon_curve(1.51, 0.65, opacus.accountants.PRVAccountant())
print({k: a[-1] for k, a in epsilon_data.items()})
plot_epsilon(epsilon_data)

##### page break

### Accuracy between epsilon consumption strategies
tune_layers: 1, learning_rate: 0.01, batch_size: 32, max_grad_norm: 2

As expected, during normal rounds all DPA-DP strategies show slightly worse but comparable accuracy to the fixed baseline. After simulated disaster onset after round 180, the low and medium strategies finish with better accuracy than the fixed baseline. The high-profile strategy 0.65 is notably performing worse in both phases.

In [ ]:
flwr_accuracy("test3/flwr", 160)

In [ ]:
flwr_accuracy("no_phase/no_phase", 160)

##### page break

### Accuracy difference
The diagram shows the accuracy of each profile relative to the fixed baseline, with deviation across
random seeds. The low 0.85 and medium 0.75 profiles show a consistent positive gap after disaster onset, with error bars that remain mostly above zero, indicating the improvement is unlikely
to be an artifact of random seed choice.

The table shows the mean and deviation values of accuracy, together with mean and deviation values relative to the fixed baseline (diff, dev_diff) over the last 10 rounds calculated for all classes
and the classes of simulated disaster group ("bird", "deer") budget3_sub.
High profile shows better accuracy for disaster group for the price of overall accuracy, while low and medium profiles show better result both in disaster group and the whole set.

##### page break

In [ ]:
flwr_accuracy_error("test3/flwr", 160)

In [ ]:
flwr_accuracy_error("no_phase/no_phase", 160)

##### page break

In [ ]:
accuracy_summary("test3/flwr")

In [ ]:
accuracy_summary("test3/flwr_sub")

In [ ]:
accuracy_summary("no_phase/no_phase")

In [ ]:
accuracy_summary("no_phase/no_phase_sub")

##### page break

### Simulated disaster accuracy between strategies
Accuracy computed separately for the labels "bird" and "deer", since they represent disaster images and matter most requiring separate attention. Accuracy is zero for all strategies up to round 180, as the model hasn't yet seen these images. After round 180, all profiles show distinguishably better accuracy than the baseline.


In [ ]:
flwr_accuracy("test3/flwr_sub", 160)

In [ ]:
flwr_accuracy("no_phase/no_phase_sub", 160)

##### page break

### Loss between strategies
Loss is comparable (roughly 4-5) across all strategies except high 0.65, which is around 6. Together with the accuracy drop, this indicates the 1.53 noise multiplier was high enough to prevent the model from learning normal images.

In [ ]:
flwr_accuracy("test3/flwr", 160, "loss")

In [ ]:
flwr_accuracy("no_phase/no_phase", 160, "loss")

##### page break

### Confusion matrixes
The confusion matrixes show a consistent error pattern across all four strategies.
Like, "bird" is frequently confused with "dog" and "deer", and "deer" is often confused with "horse". The reason is visual similarity between these classes rather than something specific to the disaster-phase setup. This pattern observed in all four profiles, suggesting that DPA-DP budget allocation changes the overall accuracy and not the kind of errors the model makes.

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(11, 4))
confusion_matrix("fixed", ax1)
confusion_matrix("dpa-dp-0.85", ax2)

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(11, 4))
confusion_matrix("dpa-dp-0.75", ax1)
confusion_matrix("dpa-dp-0.65", ax2)